# Camada ouro: modelagem

Este notebook cria as tabelas da camada ouro (`pospucrio.gold`) a partir da camada prata. É o terceiro passo do pipeline, depois de `Bronze_Ingestao` e `Silver_Limpeza`.

**O que ele faz:** monta as duas tabelas do modelo definido no `00_Pre_Projeto`, aplicando as regras de negócio, e registra no Unity Catalog a descrição de cada tabela e de cada coluna, com o domínio de valores.

**Modelo:** duas tabelas largas (modelo flat), uma por granularidade.
- `gold.pedidos`: uma linha por pedido, a perspectiva do comprador.
- `gold.itens`: uma linha por item vendido, a perspectiva do vendedor.

**Como rodar:** do início ao fim, depois do `Silver_Limpeza`. Todas as células usam `CREATE OR REPLACE`, então a reexecução sempre produz o mesmo resultado.

**Validação:** as verificações de que as tabelas ficaram corretas estão no notebook `Analises_Prata-Ouro`.

## 0. Esquema da camada ouro

In [0]:
CREATE SCHEMA IF NOT EXISTS pospucrio.gold;

## 1. gold.pedidos

**Granularidade:** um pedido por linha. Responde às perguntas sobre o comprador: compras e gasto por estado, notas, método de pagamento, forma de pagamento e evolução no tempo.

A consulta é montada em etapas (CTEs). Cada informação que vem de uma tabela com granularidade diferente da do pedido (pagamentos e avaliações podem ter várias linhas por pedido) é **reduzida a uma linha por pedido antes do join**. Juntar primeiro e agregar depois multiplicaria o valor pago pelo número de pagamentos ou de avaliações, sem nenhum aviso.

**Etapa `pagamentos`: valor e parcelamento.**
- `valor_pago` soma todos os pagamentos: produtos, frete, juros e vouchers.
- `max_parcelas` guarda o maior número de parcelas entre os pagamentos que não são voucher. Usa-se o máximo, e não a soma: dois cartões pagos em uma parcela cada são uma compra à vista.
- `parcelas_invalidas` conta pagamentos sem voucher cujo número de parcelas é nulo (zero na origem, anulado na prata).
- `nao_definidos` conta pagamentos com tipo `not_defined`.

**Etapa `metodo`: método principal.** Soma o valor pago por meio de pagamento dentro de cada pedido e ordena: primeiro os meios que não são voucher, depois pelo maior valor. O primeiro da ordem é o método principal. Com isso:
- voucher combinado com outro meio: vale o outro meio;
- pedido pago só com voucher: vale o voucher, porque é o único;
- dois meios que não são voucher: vale o de maior valor;
- o nome do meio entra como último critério apenas para garantir resultado determinístico em empate.

**Etapa `avaliacao`: uma nota por pedido.** Numera as avaliações de cada pedido da mais recente para a mais antiga: primeiro pela data de resposta do cliente, depois pela data de envio da pesquisa, e por fim pelo maior `review_id`. Fica a de número 1.

**Etapa `fretes`: frete do pedido.** Soma o frete de todos os itens de cada pedido. Como os itens têm várias linhas por pedido, a soma é feita antes do join, pelo mesmo motivo dos pagamentos.

**Consulta final:**
- `ano`, `mes`, `semestre` e `trimestre` são derivados da data de compra. O mês não estava no pré-projeto; foi acrescentado porque as comparações entre anos usam janeiro a agosto.
- `forma_pagamento` fica nula quando o pedido não tem pagamento, tem pagamento `not_defined` ou tem número de parcelas inválido. Nos demais casos, é "parcelado" se o maior número de parcelas passa de 1, e "à vista" caso contrário, inclusive nos pedidos pagos só com voucher.
- `frete_total` e `dias_entrega` não estavam no modelo do pré-projeto. Foram acrescentados durante a análise final, quando a hipótese de que a proximidade entre comprador e vendedor influencia as compras exigiu medir frete e prazo de entrega por estado. `dias_entrega` é o número de dias entre a compra e a entrega, e fica nulo para pedidos não entregues. `frete_total` fica nulo para os pedidos sem itens registrados.
- `metodo_principal` traduz os códigos da origem para português, como as categorias. `not_defined` vira nulo: não é um meio de pagamento, é a ausência de um.
- Todos os status são mantidos. O filtro de pedidos entregues é aplicado nas consultas de análise, porque as perguntas de avaliação precisam também dos não entregues.
- Pedidos a partir de janeiro de 2017. Os de 2016 são excluídos pelo baixo volume e pela lacuna na série.

In [0]:
CREATE OR REPLACE TABLE pospucrio.gold.pedidos AS
WITH pagamentos AS (
  -- uma linha por pedido: valor total e dados para a forma de pagamento
  SELECT
    order_id,
    SUM(payment_value)                                                      AS valor_pago,
    MAX(CASE WHEN payment_type <> 'voucher' THEN payment_installments END)  AS max_parcelas,
    COUNT_IF(payment_type <> 'voucher' AND payment_installments IS NULL)    AS parcelas_invalidas,
    COUNT_IF(payment_type = 'not_defined')                                  AS nao_definidos
  FROM pospucrio.silver.order_payments
  GROUP BY order_id
),
metodo AS (
  -- uma linha por pedido: o meio de pagamento principal
  SELECT order_id, payment_type
  FROM (
    SELECT
      order_id,
      payment_type,
      ROW_NUMBER() OVER (
        PARTITION BY order_id
        ORDER BY CASE WHEN payment_type = 'voucher' THEN 1 ELSE 0 END,  -- voucher por último
                 valor_meio DESC,                                         -- maior valor primeiro
                 payment_type                                             -- desempate determinístico
      ) AS ordem
    FROM (
      SELECT order_id, payment_type, SUM(payment_value) AS valor_meio
      FROM pospucrio.silver.order_payments
      GROUP BY order_id, payment_type
    )
  )
  WHERE ordem = 1
),
avaliacao AS (
  -- uma linha por pedido: a avaliação mais recente
  SELECT order_id, review_score
  FROM (
    SELECT
      order_id,
      review_score,
      ROW_NUMBER() OVER (
        PARTITION BY order_id
        ORDER BY review_answer_timestamp DESC,
                 review_creation_date    DESC,
                 review_id               DESC
      ) AS ordem
    FROM pospucrio.silver.order_reviews
  )
  WHERE ordem = 1
),
fretes AS (
  -- uma linha por pedido: frete somado dos itens
  SELECT order_id, SUM(freight_value) AS frete_total
  FROM pospucrio.silver.order_items
  GROUP BY order_id
)
SELECT
  o.order_id,
  o.order_purchase_timestamp                                   AS data_compra,
  year(o.order_purchase_timestamp)                             AS ano,
  month(o.order_purchase_timestamp)                            AS mes,
  CASE WHEN month(o.order_purchase_timestamp) <= 6 THEN 1 ELSE 2 END AS semestre,
  quarter(o.order_purchase_timestamp)                          AS trimestre,
  c.customer_state                                             AS estado_cliente,
  o.order_status                                               AS status,
  pg.valor_pago,
  f.frete_total,
  datediff(o.order_delivered_customer_date, o.order_purchase_timestamp) AS dias_entrega,   -- nulo se não entregue
  CASE
    WHEN pg.order_id IS NULL        THEN NULL          -- pedido sem pagamento
    WHEN pg.nao_definidos > 0       THEN NULL          -- pagamento not_defined
    WHEN pg.parcelas_invalidas > 0  THEN NULL          -- número de parcelas inválido na origem
    WHEN pg.max_parcelas > 1        THEN 'parcelado'
    ELSE 'à vista'                                     -- inclui pedidos pagos só com voucher
  END                                                          AS forma_pagamento,
  CASE m.payment_type
    WHEN 'credit_card' THEN 'cartão de crédito'
    WHEN 'debit_card'  THEN 'cartão de débito'
    WHEN 'boleto'      THEN 'boleto'
    WHEN 'voucher'     THEN 'voucher'
  END                                                          AS metodo_principal,   -- not_defined vira nulo
  a.review_score                                               AS nota
FROM pospucrio.silver.orders o
LEFT JOIN pospucrio.silver.customers c ON o.customer_id = c.customer_id
LEFT JOIN pagamentos pg                ON o.order_id    = pg.order_id
LEFT JOIN metodo m                     ON o.order_id    = m.order_id
LEFT JOIN avaliacao a                  ON o.order_id    = a.order_id
LEFT JOIN fretes f                     ON o.order_id    = f.order_id
WHERE o.order_purchase_timestamp >= '2017-01-01';

### 1.1 Catálogo de dados: gold.pedidos

Registra no Unity Catalog a descrição da tabela e de cada coluna, com o domínio de valores e a origem. Essas descrições aparecem no Catalog Explorer para qualquer pessoa que abra a tabela.

In [0]:
CREATE OR REPLACE TABLE pospucrio.gold.itens AS
SELECT
  i.order_id,
  i.order_item_id,
  o.order_purchase_timestamp                                         AS data_compra,
  year(o.order_purchase_timestamp)                                   AS ano,
  month(o.order_purchase_timestamp)                                  AS mes,
  CASE WHEN month(o.order_purchase_timestamp) <= 6 THEN 1 ELSE 2 END AS semestre,
  quarter(o.order_purchase_timestamp)                                AS trimestre,
  c.customer_state                                                   AS estado_cliente,
  s.seller_state                                                     AS estado_vendedor,
  coalesce(replace(p.product_category_name, '_', ' '), 'sem categoria') AS categoria,
  i.price                                                            AS preco,          -- sem frete
  o.order_status                                                     AS status
FROM pospucrio.silver.order_items i
JOIN      pospucrio.silver.orders    o ON i.order_id    = o.order_id
LEFT JOIN pospucrio.silver.customers c ON o.customer_id = c.customer_id
LEFT JOIN pospucrio.silver.sellers   s ON i.seller_id   = s.seller_id
LEFT JOIN pospucrio.silver.products  p ON i.product_id  = p.product_id
WHERE o.order_purchase_timestamp >= '2017-01-01';